# To do:

- Add markdowns
- Refactor plotting function to reuse
- Organize notebook
- Compare SMOTE with Undersampling (?)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from itertools import product
from sklearn.datasets import make_classification
import random
import plotly.io as pio
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import matplotlib.gridspec as gridspec

from plotly.subplots import make_subplots

# Metrics
from scipy.stats import ks_2samp
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, log_loss, average_precision_score
) # roc_curve, precision_recall_curve


from sklearn.linear_model import LogisticRegression

In [2]:

def calculate_ks(y, y_prob):
    if isinstance(y, pd.DataFrame):
        y = y.values

    if isinstance(y, np.ndarray):
        if len(y.shape) > 1:
            y = y.flatten()

    true_labels = y
    predicted_proba = y_prob

    data = pd.DataFrame({
        'true_labels': true_labels,
        'predicted_proba': predicted_proba
    })

    # Separate the probabilities into two groups based on true labels
    positive_proba = data[data['true_labels'] == 1]['predicted_proba']
    negative_proba = data[data['true_labels'] == 0]['predicted_proba']

    # Compute KS statistic and p-value using ks_2samp
    ks_stat, p_value = ks_2samp(positive_proba, negative_proba)

    return ks_stat, p_value


def calculate_model_metrics_datasets(dataset_dict):
    roc_auc, accuracy, f1, precision, recall, ks = [], [], [], [], [], []
    specificity, mcc, logloss, pr_auc, balanced_accuracy = [], [], [], [], []
    for set_name in dataset_dict:
        # x_ = dataset_dict[set_name]['x']
        y_ = dataset_dict[set_name]['y']
        y_pred = dataset_dict[set_name]['y_pred']
        y_prob = dataset_dict[set_name]['y_prob']

        tn, fp, fn, tp = confusion_matrix(y_, y_pred).ravel()

        roc_auc.append(roc_auc_score(y_, y_pred))
        accuracy.append(accuracy_score(y_, y_pred))
        f1.append(f1_score(y_, y_pred))
        precision.append(precision_score(y_, y_pred))
        recall.append(recall_score(y_, y_pred))
        ks.append(calculate_ks(y_, y_prob)[0])

        specificity.append(tn / (tn + fp))  # Specificity (True Negative Rate)
        mcc.append((tp * tn - fp * fn) / 
                   ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5)  # Matthews Correlation Coefficient
        logloss.append(log_loss(y_, y_prob))  # Log Loss
        pr_auc.append(average_precision_score(y_, y_prob))  # Precision-Recall AUC
        balanced_accuracy.append((recall[-1] + specificity[-1]) / 2)  # Balanced Accuracy
    metrics = pd.DataFrame({
        'ROC AUC': roc_auc,
        'Accuracy': accuracy,
        'F1 Score': f1,
        'Precision': precision,
        'Recall': recall,
        'KS': ks,
        'Specificity': specificity,
        'MCC': mcc,
        'Log Loss': logloss,
        'PR AUC': pr_auc,
        'Balanced Accuracy': balanced_accuracy,
    }, index=dataset_dict.keys())
    return metrics



# Dataset

In [3]:
# metric_name_list = ['Accuracy', 'F1 Score', 'Precision', 'Recall', 'Balanced Accuracy', 'Specificity']
metric_name_list = ['Accuracy', 'Precision', 'Recall', 'Balanced Accuracy', 'Specificity']


In [4]:

def undersample_to_ratio(X, y, target_ratio=0.8, random_state=None):
    """
    Undersamples the majority class (label 0) to reach a specified class imbalance ratio.
    
    Parameters:
    - X: Features (pd.DataFrame or np.ndarray)
    - y: Labels (pd.Series, list, or np.ndarray)
    - target_ratio: Desired ratio of majority class (e.g. 0.8 for 80%)
    - random_state: Optional random seed
    
    Returns:
    - X_resampled, y_resampled: Undersampled feature matrix and labels
    """
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X)
    if not isinstance(y, pd.Series):
        y = pd.Series(y)

    # Separate majority and minority
    X_majority = X[y == 0]
    X_minority = X[y == 1]
    
    y_majority = y[y == 0]
    y_minority = y[y == 1]

    n_minority = len(y_minority)

    # Compute how many majority samples are needed for the desired ratio
    n_majority_desired = int((target_ratio / (1 - target_ratio)) * n_minority)

    # Undersample the majority class
    X_majority_downsampled = X_majority.sample(n=n_majority_desired, random_state=random_state)
    y_majority_downsampled = y_majority.loc[X_majority_downsampled.index]

    # Combine
    X_resampled = pd.concat([X_majority_downsampled, X_minority], axis=0).reset_index(drop=True)
    y_resampled = pd.concat([y_majority_downsampled, y_minority], axis=0).reset_index(drop=True)

    return X_resampled, y_resampled


def undersample_to_ratio_numpy(X, y, target_ratio=0.8, random_state=None):
    """
    Undersample majority class (label 0) in a binary classification setting to achieve desired class ratio.
    
    Parameters:
    - X: np.ndarray of shape (n_samples, n_features)
    - y: np.ndarray of shape (n_samples,) or (n_samples, 1)
    - target_ratio: Desired proportion of majority class (e.g. 0.7 for 70% label 0)
    - random_state: Optional seed for reproducibility
    
    Returns:
    - X_resampled: np.ndarray
    - y_resampled: np.ndarray (shape (n_samples_resampled, 1))
    """
    if y.ndim == 2:
        y = y.ravel()  # Flatten (n,1) -> (n,)
    
    # Indices for each class
    idx_majority = np.where(y == 0)[0]
    idx_minority = np.where(y == 1)[0]
    
    n_minority = len(idx_minority)
    
    # Desired number of majority samples to reach target ratio
    n_majority_desired = int((target_ratio / (1 - target_ratio)) * n_minority)
    
    if n_majority_desired > len(idx_majority):
        raise ValueError(f"Target ratio {target_ratio} requires {n_majority_desired} majority samples, "
                         f"but only {len(idx_majority)} available. Choose a higher imbalance.")
    
    rng = np.random.default_rng(random_state)
    idx_majority_downsampled = rng.choice(idx_majority, size=n_majority_desired, replace=False)

    # Combine indices
    idx_resampled = np.concatenate([idx_majority_downsampled, idx_minority])
    rng.shuffle(idx_resampled)  # Shuffle for good measure

    X_resampled = X[idx_resampled]
    y_resampled = y[idx_resampled].reshape(-1, 1)  # Return shape (n_samples_resampled, 1)

    return X_resampled, y_resampled


def create_sample_dataset(n_samples, weights=[0.9, 0.1], random_seed=42):
    random.seed(random_seed)

    n_small_categorical_features = 2
    n_large_categorical_features = 3
    n_numerical_features = 3

    n_categorical_features = n_small_categorical_features + n_large_categorical_features
    n_features = n_numerical_features + n_categorical_features
    columns = (
        [f'num_feature_{i}' for i in range(n_numerical_features)]
        + [f'cat_small_feature_{i}' for i in range(n_small_categorical_features)]
        + [f'cat_large_feature_{i}' for i in range(n_large_categorical_features)]
        + ['target']
    )

    # --- Dataset Creation ---
    X, y = make_classification(n_samples=n_samples, n_features=n_features, weights=weights, random_state=random_seed, flip_y=0)
    y = y.reshape(-1, 1)
    df = pd.DataFrame(np.concatenate([X, y], axis=1), columns=columns)

    # --- Numerical Features ---

    # --- Small Categorical Features ---
    for i in range(n_small_categorical_features):
        col = f"cat_small_feature_{i}"
        n_bins = random.randint(2, 5)
        df[col] = pd.cut(df[col], bins=n_bins, labels=[f'Small_{j}' for j in range(1, n_bins + 1)])

    # --- Large Categorical Features ---
    for i in range(n_large_categorical_features):
        col = f"cat_large_feature_{i}"
        n_bins = random.randint(20, 50)
        df[col] = pd.cut(df[col], bins=n_bins, labels=[f'Large_{j}' for j in range(1, n_bins + 1)])

    return df


# Baseline

Completely balanced dataset

In [5]:

def split_and_process(df):

    # --- Step 2: Train-test split (keep test set imbalanced) ---
    df_train, df_test = train_test_split(
        df, test_size=0.30, random_state=42, stratify=df[target]
    )

    # --- Step 3: Define preprocessing
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('cat', categorical_transformer, categorical_cols)
    ], remainder='passthrough')

    target_transformer = Pipeline(steps=[
        ('encoder', OrdinalEncoder())
    ])

    # --- Step 4: Preprocess training data before rebalancing ---
    X_train_proc = preprocessor.fit_transform(df_train[features])
    y_train_proc = target_transformer.fit_transform(df_train[[target]])

    return (
        df_train, df_test,
        X_train_proc, y_train_proc,
        preprocessor, target_transformer
    )



In [6]:

# --- Step 1: Load dataset ---
df = create_sample_dataset(n_samples=1000, weights=[0.5, 0.5])
target = 'target'
categorical_cols = [col for col in df.columns if col.startswith('cat_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
features = numerical_cols + categorical_cols

# --- Step 2: Train-test split (keep test set imbalanced) ---
df_train, df_test = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df[target]
)

# --- Step 3: Define preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_cols)
], remainder='passthrough')

target_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder())
])

# --- Step 4: Preprocess training data before rebalancing ---
X_train_proc = preprocessor.fit_transform(df_train[features])
y_train_proc = target_transformer.fit_transform(df_train[[target]])


# --- Step 6: Train model on resampled data ---
model = LogisticRegression()
model.fit(X_train_proc, y_train_proc.flatten())

# --- Step 7: Final pipeline for inference
inference_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

predictions_dict = {
    "Train (Balanced)": {
        "y": y_train_proc,
        "y_pred": model.predict(X_train_proc),
        "y_prob": model.predict_proba(X_train_proc)[:, 1]
    },
    "Test (Imbalanced)": {
        "y": df_test[target],
        "y_pred": inference_pipeline.predict(df_test[features]),
        "y_prob": inference_pipeline.predict_proba(df_test[features])[:, 1]
    },
}

metrics_df_ = calculate_model_metrics_datasets(predictions_dict)
metrics_df_

# Accuracy, F1 Score, Precision and Recall should be around ~85%

,ROC AUC,Accuracy,F1 Score,Precision,Recall,KS,Specificity,MCC,Log Loss,PR AUC,Balanced Accuracy
Train (Balanced),0.868571,0.868571,0.871148,0.854396,0.888571,0.742857,0.848571,0.737733,0.317746,0.938304,0.868571
Test (Imbalanced),0.883333,0.883333,0.883721,0.880795,0.886667,0.773333,0.880000,0.766684,0.332855,0.931782,0.883333


# Training without rebalancing the train set

In [7]:

weights_list = [[.9], [.8], [.7], [.6], [.5], [.4], [.3], [.2], [.1], ]

for weights in weights_list:
    df = create_sample_dataset(n_samples=1000, weights=weights)
    print(f"{(df[target]==1).sum() / (len(df)):.0} - {(df[target]==0).sum() / (len(df)):.0}")


0.1 - 0.9
0.2 - 0.8
0.3 - 0.7
0.4 - 0.6
0.5 - 0.5
0.6 - 0.4
0.7 - 0.3
0.8 - 0.2
0.9 - 0.1


In [8]:
metrics_df = pd.DataFrame()
train_balances_df = pd.DataFrame()

test_balances_list = []
train_balances_list = []
metrics_list = []

weights_list = [[.9], [.8], [.7], [.6], [.5], [.4], [.3], [.2], [.1], ]

for weights in weights_list:

    # --- Step 1: Load dataset ---
    df = create_sample_dataset(n_samples=1000, weights=weights)
    target = 'target'
    categorical_cols = [col for col in df.columns if col.startswith('cat_')]
    numerical_cols = [col for col in df.columns if col.startswith('num_')]
    features = numerical_cols + categorical_cols

    # --- Step 2: Train-test split (keep test set imbalanced) ---
    df_train, df_test = train_test_split(
        df, test_size=0.30, random_state=42, stratify=df[target]
    )

    # --- Step 3: Define preprocessing
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('cat', categorical_transformer, categorical_cols)
    ], remainder='passthrough')

    target_transformer = Pipeline(steps=[
        ('encoder', OrdinalEncoder())
    ])

    # --- Step 4: Preprocess training data before rebalancing ---
    X_train_proc = preprocessor.fit_transform(df_train[features])
    y_train_proc = target_transformer.fit_transform(df_train[[target]])


    # --- Step 6: Train model on resampled data ---
    model = LogisticRegression()
    model.fit(X_train_proc, y_train_proc.flatten())

    # --- Step 7: Final pipeline for inference
    inference_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    predictions_dict = {
        "Train (Balanced)": {
            "y": y_train_proc,
            "y_pred": model.predict(X_train_proc),
            "y_prob": model.predict_proba(X_train_proc)[:, 1]
        },
        "Test (Imbalanced)": {
            "y": target_transformer.transform(df_test[[target]]),
            "y_pred": inference_pipeline.predict(df_test[features]),
            "y_prob": inference_pipeline.predict_proba(df_test[features])[:, 1]
        },
    }

    metrics_df_ = calculate_model_metrics_datasets(predictions_dict)
    metrics_df_['Target proportion'] = int((1 - weights[0])*100)
    metrics_list.append(metrics_df_)
    metrics_df = pd.concat([metrics_df, metrics_df_])

    train_balances_list.append(df_train[target].value_counts())
    test_balances_list.append(df_test[target].value_counts())

    df_ = df_train[target].value_counts().to_frame().reset_index()
    df_['Target proportion'] = int((1 - weights[0])*100)
    train_balances_df = pd.concat([train_balances_df, df_])


In [9]:
# fig, axes = plt.subplots(nrows=5, figsize=(6,8))

# for metric_, ax in zip(['Accuracy', 'F1 Score', 'Precision', 'Recall', 'Balanced Accuracy', 'Specificity'], axes):
#     sns.barplot(
#         metrics_df[metrics_df.index.str.contains('Test')],
#         x='Target proportion',
#         y=metric_,
#         ax=ax
#     )

# plt.tight_layout()

In [10]:
# (metrics_df[metrics_df.index.str.contains('Test')]
#  .melt(id_vars=['Target proportion'])
#  )

In [11]:

# fig = plt.figure(figsize=(8, 6))
# gs = gridspec.GridSpec(nrows=2, ncols=1, height_ratios=[.7, .3])

# # Create subplots in the custom grid
# ax1 = fig.add_subplot(gs[0, 0])
# ax2 = fig.add_subplot(gs[1, 0])


# aff = (metrics_df[metrics_df.index.str.contains('Test')]
#  [metric_name_list + ['Target proportion']]
#  .melt(id_vars=['Target proportion'])
#  )

# sns.lineplot(
#     aff,
#     x='Target proportion',
#     y='value',
#     hue='variable',
#     ax=ax1
# )

# sns.barplot(train_balances_df, x='Target proportion', y='count', hue='target', errorbar=None, ax=ax2)
# plt.tight_layout()

import plotly.graph_objects as go
import pandas as pd

# Prepare the data
aff = (metrics_df[metrics_df.index.str.contains('Test')]
       [metric_name_list + ['Target proportion']]
       .melt(id_vars=['Target proportion'])
)

# Line plot (for ax1 in seaborn)
line_fig = go.Figure()

for variable in aff['variable'].unique():
    line_fig.add_trace(go.Scatter(
        x=aff[aff['variable'] == variable]['Target proportion'],
        y=aff[aff['variable'] == variable]['value'],
        mode='lines',
        name=variable
    ))

line_fig.update_layout(
    title='Line Plot',
    xaxis_title='Target Proportion',
    yaxis_title='Value',
    height=600,
)

# Bar plot (for ax2 in seaborn)
bar_fig = go.Figure()

for target in train_balances_df['target'].unique():
    bar_fig.add_trace(go.Bar(
        x=train_balances_df[train_balances_df['target'] == target]['Target proportion'],
        y=train_balances_df[train_balances_df['target'] == target]['count'],
        name=target,
        marker=dict(line=dict(width=0))  # Equivalent to errorbar=None
    ))

bar_fig.update_layout(
    title='Bar Plot',
    xaxis_title='Target Proportion',
    yaxis_title='Count',
    height=300,
)

# Combine the two subplots into a single figure using plotly subplots
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=('Line Plot', 'Bar Plot'),
    row_heights=[0.7, 0.3]
)

# Add line plot to the first subplot
for trace in line_fig.data:
    fig.add_trace(trace, row=1, col=1)

# Add bar plot to the second subplot
for trace in bar_fig.data:
    fig.add_trace(trace, row=2, col=1)

# Update layout for overall figure
fig.update_layout(
    title='Combined Plot',
    showlegend=True
)

fig.show()


In [12]:
# # fig, axes = plt.subplots(figsize=(8,4))

# fig = plt.figure(figsize=(8, 6))
# gs = gridspec.GridSpec(nrows=2, ncols=1, height_ratios=[.7, .3])

# # Create subplots in the custom grid
# ax1 = fig.add_subplot(gs[0, 0])
# ax2 = fig.add_subplot(gs[1, 0])


# aff = (metrics_df[metrics_df.index.str.contains('Train')]
#  [metric_name_list + ['Target proportion']]
#  .melt(id_vars=['Target proportion'])
#  )

# sns.lineplot(
#     aff,
#     x='Target proportion',
#     y='value',
#     hue='variable',
#     ax=ax1
# )

# sns.barplot(train_balances_df, x='Target proportion', y='count', hue='target', errorbar=None, ax=ax2)

# ax1.set_title('Training Metrics')
# ax2.set_title('Class proportions on the training set')

# plt.tight_layout()

In [13]:

# # Create the figure and gridspec layout
# n = len(weights_list)

# fig = plt.figure(figsize=(8, 10))
# gs = gridspec.GridSpec(n, 3, width_ratios=[.2, .2, .6])  # First column 20%, second column 80%

# # Create subplots in the custom grid
# axes_col1 = [fig.add_subplot(gs[i, 0]) for i in range(n)]  # First column (n=5)
# axes_col2 = [fig.add_subplot(gs[i, 1]) for i in range(n)]  # Second column (n=5)
# axes_col3 = [fig.add_subplot(gs[i, 2]) for i in range(n)]  # Second column (n=5)

# for train_balances, ax in zip(train_balances_list, axes_col1):
#     train_balances.plot.bar(ax=ax)

# for test_balances, ax in zip(test_balances_list, axes_col2):
#     test_balances.plot.bar(ax=ax)


# for metrics_df, ax in zip(metrics_list, axes_col3):
#     # df_ = metrics_df[metrics_df.index.str.contains('Test')]
#     df_ = (metrics_df [metrics_df.index.str.contains('Test')]
#         .reset_index(names='Data set') [['Data set'] + metric_name_list]
#         .melt(id_vars='Data set'))
#     sns.barplot(
#         df_,
#         x='variable',
#         y='value',
#         ax=ax
#     )

# # Now, you can plot on these axes as you normally would
# # for i, ax in enumerate(axes):
# #     ax.plot(range(10), label=f"Plot {i + 1}")
# #     ax.set_title(f"Plot {i + 1}")
# #     ax.legend()


# # for metric_, ax in zip(['Accuracy', 'F1 Score', 'Precision', 'Recall', 'Balanced Accuracy'], axes):
# #     sns.barplot(
# #         metrics_df[metrics_df.index.str.contains('Test')],
# #         x='Target proportion',
# #         y=metric_,
# #         ax=ax
# #     )

# plt.tight_layout()

In [14]:

# # --- Step 3: Define preprocessing for SMOTE (OneHot + Imputation) ---
# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
# ])

# preprocessor = ColumnTransformer(transformers=[
#     ('cat', categorical_transformer, categorical_cols)
# ], remainder='passthrough')

# target_transformer = Pipeline(steps=[
#     ('encoder', OrdinalEncoder())
# ])

# # --- Step 4: Preprocess training data before SMOTE ---
# # df_train_proc = preprocessor.fit_transform(df_train[features])

# X_train_proc = preprocessor.fit_transform(df_train[features])
# y_train_proc = target_transformer.fit_transform(df_train[[target]])


In [15]:
# from imblearn.over_sampling import SMOTE
# import numpy as np

# metrics_df = pd.DataFrame()
# for target_ratio in [.9, .8, .7, .6, .5]:

#     # --- Step 6: Train model on resampled data ---
#     model = LogisticRegression()
#     model.fit(X_train_proc, y_train_proc.flatten())

#     # --- Step 7: Final pipeline for inference
#     inference_pipeline = Pipeline(steps=[
#         ('preprocessor', preprocessor),
#         ('classifier', model)
#     ])

#     predictions_dict = {
#         "Train (Balanced)": {
#             "y": y_train_proc,
#             "y_pred": model.predict(X_train_proc),
#             "y_prob": model.predict_proba(X_train_proc)[:, 1]
#         },
#         "Test (Imbalanced)": {
#             "y": df_test[target],
#             "y_pred": inference_pipeline.predict(df_test[features]),
#             "y_prob": inference_pipeline.predict_proba(df_test[features])[:, 1]
#         },
#     }

#     metrics_df_ = calculate_model_metrics_datasets(predictions_dict)
#     metrics_df_['Target proportion'] = target_ratio*100
#     metrics_df = pd.concat([metrics_df, metrics_df_])

#     print(f"{(y_train_proc==1).sum() / (len(y_train_proc)):.3f}")
#     print(f"{(y_train_res==1).sum() / (len(y_train_res)):.3f}")
#     print()

# Undersampling

In [16]:

# --- Step 1: Load dataset ---
df = create_sample_dataset(n_samples=1000, weights=[0.89, 0.09])
target = 'target'
categorical_cols = [col for col in df.columns if col.startswith('cat_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
features = numerical_cols + categorical_cols

# --- Step 2: Train-test split (keep test set imbalanced) ---
df_train, df_test = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df[target]
)

# --- Step 3: Define preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_cols)
], remainder='passthrough')

target_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder())
])

# --- Step 4: Preprocess training data before rebalancing ---
X_train_proc = preprocessor.fit_transform(df_train[features])
y_train_proc = target_transformer.fit_transform(df_train[[target]])


In [29]:
# predictions_list = []
metrics_df = pd.DataFrame()
train_balances_df = pd.DataFrame()

test_balances_list = []
train_balances_list = []
metrics_list = []


for target_ratio in [.9, .8, .7, .6, .5, .4, .3, .2, .1]:
    X_train_res, y_train_res = undersample_to_ratio_numpy(
        X_train_proc, y_train_proc,
        target_ratio=target_ratio, random_state=42
    )

    # --- Step 6: Train model on resampled data ---
    model = LogisticRegression() # solver='liblinear', class_weight='balanced', random_state=42
    model.fit(X_train_res, y_train_res.flatten())

    # --- Step 7: Final pipeline for inference
    inference_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    predictions_dict = {
        "Train (Balanced)": {
            "y": y_train_res,
            "y_pred": model.predict(X_train_res),
            "y_prob": model.predict_proba(X_train_res)[:, 1]
        },
        "Test (Imbalanced)": {
            "y": df_test[target],
            "y_pred": inference_pipeline.predict(df_test[features]),
            "y_prob": inference_pipeline.predict_proba(df_test[features])[:, 1]
        },
    }

    metrics_df_ = calculate_model_metrics_datasets(predictions_dict)
    # metrics_df_['Target proportion'] = target_ratio*100
    metrics_df_['Target proportion'] = int((1 - target_ratio)*100)
    metrics_list.append(metrics_df_)
    metrics_df = pd.concat([metrics_df, metrics_df_])

    test_balances_list.append(df_test[target].value_counts())

    vc = pd.DataFrame(y_train_res, columns=[target]).value_counts()
    train_balances_list.append(vc)
    df_ = vc.to_frame().reset_index()
    df_['Target proportion'] = int((1 - target_ratio)*100)
    train_balances_df = pd.concat([train_balances_df, df_])


    # print(f"{(y_train_proc==1).sum() / (len(y_train_proc)):.3f} - {(y_train_proc==0).sum() / (len(y_train_proc)):.3f}")
    # print(f"{(y_train_res==1).sum() / (len(y_train_res)):.3f} - {(y_train_res==0).sum() / (len(y_train_res)):.3f}")
    # print()
    # display(metrics_df)




target
0.0       630
1.0        70
Name: count, dtype: int64

In [30]:

# Prepare the data
aff = (metrics_df[metrics_df.index.str.contains('Test')]
       [metric_name_list + ['Target proportion']]
       .melt(id_vars=['Target proportion'])
)

# Line plot (for ax1 in seaborn)
line_fig = go.Figure()

for variable in aff['variable'].unique():
    line_fig.add_trace(go.Scatter(
        x=aff[aff['variable'] == variable]['Target proportion'],
        y=aff[aff['variable'] == variable]['value'],
        mode='lines',
        name=variable
    ))

line_fig.update_layout(
    title='Line Plot',
    xaxis_title='Target Proportion',
    yaxis_title='Value',
    height=600,
)

# Bar plot (for ax2 in seaborn)
bar_fig = go.Figure()

for target_ in train_balances_df['target'].unique():
    bar_fig.add_trace(go.Bar(
        x=train_balances_df[train_balances_df['target'] == target_]['Target proportion'],
        y=train_balances_df[train_balances_df['target'] == target_]['count'],
        name=target_,
        marker=dict(line=dict(width=0))  # Equivalent to errorbar=None
    ))

bar_fig.update_layout(
    title='Bar Plot',
    xaxis_title='Target Proportion',
    yaxis_title='Count',
    height=300,
)

# Combine the two subplots into a single figure using plotly subplots
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=('Line Plot', 'Bar Plot'),
    row_heights=[0.7, 0.3]
)

# Add line plot to the first subplot
for trace in line_fig.data:
    fig.add_trace(trace, row=1, col=1)

# Add bar plot to the second subplot
for trace in bar_fig.data:
    fig.add_trace(trace, row=2, col=1)

# Update layout for overall figure
fig.update_layout(
    title='Combined Plot',
    showlegend=True
)

fig.show()


In [ ]:
fig, axes = plt.subplots(nrows=6, figsize=(6,8))

for metric_, ax in zip(['Accuracy', 'F1 Score', 'Precision', 'Recall', 'Balanced Accuracy', 'Specificity'], axes):
    sns.barplot(
        metrics_df[metrics_df.index.str.contains('Test')],
        x='Target proportion',
        y=metric_,
        ax=ax
    )

plt.tight_layout()

# Synthetic Minority Over-sampling Technique (SMOTE)

In [57]:
from imblearn.over_sampling import SMOTE
import numpy as np


# predictions_list = []
metrics_df = pd.DataFrame()
train_balances_df = pd.DataFrame()
test_balances_df = pd.DataFrame()

test_balances_list = []
train_balances_list = []
metrics_list = []


def oversample_to_ratio_smote_numpy(X, y, target_ratio=0.8, random_state=None, k_neighbors=5):
    """
    Use SMOTE to oversample the minority class (label 1) to achieve the desired class ratio.
    
    Parameters:
    - X: np.ndarray of shape (n_samples, n_features)
    - y: np.ndarray of shape (n_samples,) or (n_samples, 1)
    - target_ratio: Desired proportion of majority class (e.g. 0.7 for 70% label 0)
    - random_state: Optional seed for reproducibility
    - k_neighbors: Number of nearest neighbors used in SMOTE
    
    Returns:
    - X_resampled: np.ndarray
    - y_resampled: np.ndarray (shape (n_samples_resampled, 1))
    """
    if y.ndim == 2:
        y = y.ravel()  # Flatten (n,1) → (n,)

    # Current counts
    n_maj = np.sum(y == 0)
    n_min = np.sum(y == 1)

    # Desired number of minority samples to reach target ratio
    n_min_desired = int((1 - target_ratio) / target_ratio * n_maj)

    # If there are already enough minority samples, skip resampling
    if n_min >= n_min_desired:
        print(f"Minority class already has enough samples. No oversampling needed.")
        return X, y.reshape(-1, 1)  # Return original data

    # Calculate the sampling strategy (if it exceeds 1, set it to 1)
    sampling_strategy = min(n_min_desired / n_maj, 1.0)  # Ensure it doesn't exceed 1

    smote = SMOTE(sampling_strategy=sampling_strategy, random_state=random_state, k_neighbors=k_neighbors)
    X_resampled, y_resampled = smote.fit_resample(X, y)

    # Reshape y to (n_samples, 1)
    y_resampled = y_resampled.reshape(-1, 1)

    return X_resampled, y_resampled

metrics_df = pd.DataFrame()
for target_ratio in [.9, .8, .7, .6, .5]:

    X_train_res, y_train_res = oversample_to_ratio_smote_numpy(
        X_train_proc, y_train_proc,
        target_ratio=target_ratio, random_state=42
    )

    # --- Step 6: Train model on resampled data ---
    model = LogisticRegression()
    model.fit(X_train_res, y_train_res.flatten())

    # --- Step 7: Final pipeline for inference
    inference_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    predictions_dict = {
        "Train (Balanced)": {
            "y": y_train_res,
            "y_pred": model.predict(X_train_res),
            "y_prob": model.predict_proba(X_train_res)[:, 1]
        },
        "Test (Imbalanced)": {
            "y": df_test[target],
            "y_pred": inference_pipeline.predict(df_test[features]),
            "y_prob": inference_pipeline.predict_proba(df_test[features])[:, 1]
        },
    }

    metrics_df_ = calculate_model_metrics_datasets(predictions_dict)
    metrics_df_['Target proportion'] = int((1 - target_ratio)*100)
    metrics_list.append(metrics_df_)
    metrics_df = pd.concat([metrics_df, metrics_df_])


    vc = pd.DataFrame(y_train_res, columns=[target]).value_counts()
    train_balances_list.append(vc)
    df_ = vc.to_frame().reset_index()
    df_['Target proportion'] = int((1 - target_ratio)*100)
    train_balances_df = pd.concat([train_balances_df, df_])


    test_balances_list.append(df_test[target].value_counts())

    vc = pd.DataFrame(df_test[target], columns=[target]).value_counts()
    test_balances_list.append(vc)
    df_ = vc.to_frame().reset_index()
    df_['Target proportion'] = int((1 - target_ratio)*100)
    test_balances_df = pd.concat([test_balances_df, df_])




    print(f"{(y_train_proc==1).sum() / (len(y_train_proc)):.3f}")
    print(f"{(y_train_res==1).sum() / (len(y_train_res)):.3f}")
    print()

Minority class already has enough samples. No oversampling needed.
0.100
0.100

0.100
0.199

0.100
0.300

0.100
0.400

0.100
0.500



474    0.0
560    0.0
357    0.0
242    0.0
761    0.0
      ... 
917    1.0
209    1.0
839    0.0
207    0.0
515    0.0
Name: target, Length: 300, dtype: float64

In [80]:
import plotly.colors as pcolors


def foo(metrics_df, test_balances_df, train_balances_df):

    line_plot_title = 'Model Metrics on Test Set'
    bar_plot_title = 'Target Proportions on Train Set'

    colors = pcolors.qualitative.Plotly

    # Prepare the data
    metrics_df_test = (metrics_df[metrics_df.index.str.contains('Test')]
        [metric_name_list + ['Target proportion']]
        .melt(id_vars=['Target proportion'])
    )
    metrics_df_train = (metrics_df[metrics_df.index.str.contains('Train')]
        [metric_name_list + ['Target proportion']]
        .melt(id_vars=['Target proportion'])
    )


    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=(line_plot_title, bar_plot_title),
        row_heights=[0.7, 0.3]
    )

    # --- Line plot (Train) ---
    line_fig = go.Figure()
    for i, variable in enumerate(metrics_df_train['variable'].unique()):
        line_fig.add_trace(go.Scatter(
            x=metrics_df_train[metrics_df_train['variable'] == variable]['Target proportion'],
            y=metrics_df_train[metrics_df_train['variable'] == variable]['value'],
            mode='lines+markers',
            marker=dict(color=colors[i], size=10),
            line=dict(dash='dash'),
            name=variable,
            legendgroup="group_line_1",
            legendgrouptitle_text="Train",
            visible='legendonly',
        ))

    for trace in line_fig.data:
        fig.add_trace(trace, row=1, col=1)

    # --- Line plot (Test)  ---
    line_fig = go.Figure()
    for i, variable in enumerate(metrics_df_test['variable'].unique()):
        line_fig.add_trace(go.Scatter(
            x=metrics_df_test[metrics_df_test['variable'] == variable]['Target proportion'],
            y=metrics_df_test[metrics_df_test['variable'] == variable]['value'],
            mode='lines+markers',
            marker=dict(color=colors[i], size=10),
            name=variable,
            legendgroup="group_line_2",
            legendgrouptitle_text="Test",
            
        ))

    for trace in line_fig.data:
        fig.add_trace(trace, row=1, col=1)

    # --- Bar plot ---
    bar_train_fig = go.Figure()
    for i, target_ in enumerate(train_balances_df['target'].unique()):
        bar_train_fig.add_trace(go.Bar(
            x=train_balances_df[train_balances_df['target'] == target_]['Target proportion'],
            y=train_balances_df[train_balances_df['target'] == target_]['count'],
            name=target_,
            # marker_color=colors[i],
            legendgroup="group_bar_1",
            legendgrouptitle_text="Train",
        
            # name="Train",
            # marker=dict(line=dict(width=0))  # Equivalent to errorbar=None
        ))

    for trace in bar_train_fig.data:
        fig.add_trace(trace, row=2, col=1)


    bar_test_fig = go.Figure()
    for i, target_ in enumerate(test_balances_df['target'].unique()):
        bar_test_fig.add_trace(go.Bar(
            x=test_balances_df[test_balances_df['target'] == target_]['Target proportion'],
            y=test_balances_df[test_balances_df['target'] == target_]['count'],
            name=target_,
            # marker_color=colors[i],
            legendgroup="group_bar_2",
            legendgrouptitle_text="Test",
            visible='legendonly',

            # name="test",
            # marker=dict(line=dict(width=0))  # Equivalent to errorbar=None
        ))

    for trace in bar_test_fig.data:
        fig.add_trace(trace, row=2, col=1)


    # Update layout for overall figure
    fig.update_layout(title='Combined Plot', showlegend=True)

    fig.update_yaxes(title_text="Metric value", row=1, col=1)
    fig.update_xaxes(title_text="Target Proportion", row=2, col=1)
    fig.update_yaxes(title_text="Count", row=2, col=1)

    return fig


foo(metrics_df, test_balances_df, train_balances_df)

In [ ]:
fig, axes = plt.subplots(nrows=5, figsize=(6,8))

for metric_, ax in zip(['Accuracy', 'F1 Score', 'Precision', 'Recall', 'Balanced Accuracy'], axes):
    sns.barplot(
        metrics_df[metrics_df.index.str.contains('Test')],
        x='Target proportion',
        y=metric_,
        ax=ax
    )

plt.tight_layout()

In [ ]:

# # --- Step 5: Apply SMOTE only on training data ---
# smote = SMOTE(random_state=42)
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train_preprocessed, y_train)

# # --- Optional: Check new class distribution ---
# from collections import Counter
# print("Original training target distribution:", Counter(y_train))
# print("Resampled training target distribution:", Counter(y_train_resampled))



# Measuring the effects of rebalancing on the model

Original imbalance: 95% / 5% - Baseline
Bance 1: Increase

In [ ]:

# --- Step 1: Load dataset ---
df = create_sample_dataset(n_samples=1000, weights=[0.95, 0.05])
target_col = 'target'

categorical_cols = [col for col in df.columns if col.startswith('cat_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
X = df[categorical_cols + numerical_cols]
y = df[target_col]

# --- Step 2: Train-test split (keep test set imbalanced) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Step 3: Define preprocessing (OneHot for categoricals) ---
categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_cols)
], remainder='passthrough')

# --- Step 4: Preprocess training data (before SMOTE) ---
X_train_preprocessed = preprocessor.fit_transform(X_train)

# --- Step 5: Apply SMOTE only on training data ---
sampling_strategy = {
    0: int((y_train == 0).sum()),
    1: int((y_train == 0).sum() * 1),
}
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_preprocessed, y_train)

print("Original training target distribution:", Counter(y_train))
print("Resampled training target distribution:", Counter(y_train_resampled))

# --- Step 6: Train model on resampled data ---
model = LogisticRegression(
    solver='liblinear', 
    class_weight='balanced', 
    random_state=42
)
model.fit(X_train_resampled, y_train_resampled)

# --- Step 7: Final pipeline for inference (no SMOTE, just preprocessing + model) ---
inference_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

# --- Step 8: Predict on test set (untouched data) ---
# y_pred = inference_pipeline.predict(X_test)
# y_proba = inference_pipeline.predict_proba(X_test)[:, 1]

predictions_dict = {
    "Train (Imbalanced)": {
        "y": y_train,
        "y_pred": inference_pipeline.predict(X_train),
        "y_prob": inference_pipeline.predict_proba(X_train)[:, 1]
    },
    "Train (Balanced)": {
        "y": y_train_resampled,
        "y_pred": model.predict(X_train_resampled),
        "y_prob": model.predict_proba(X_train_resampled)[:, 1]
    },
    "Test (Imbalanced)": {
        "y": y_test,
        "y_pred": inference_pipeline.predict(X_test),
        "y_prob": inference_pipeline.predict_proba(X_test)[:, 1]
    },

}

calculate_model_metrics_datasets(predictions_dict)

# The wrong way (Test on Balanced Dataset)

In [ ]:

# --- Step 1: Load dataset ---
df = create_sample_dataset(n_samples=1000, weights=[0.95, 0.05])
target_col = 'target'

categorical_cols = [col for col in df.columns if col.startswith('cat_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
feature_cols = categorical_cols + numerical_cols
X = df[feature_cols]
y = df[target_col]


# --- Step 2: Train-test split (keep test set imbalanced) ---
X_, X_test_imbal, y_, y_test_imbal = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# --- Step 2: Define preprocessing (OneHot for categoricals) ---
categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_cols)
], remainder='passthrough')  # Keep numerical features unchanged

# --- Step 3: Preprocess training data ---
X_preprocessed = preprocessor.fit_transform(X_)

# --- Step 4: Apply SMOTE on the whole data ---
sampling_strategy = {
    0: int((y_train == 0).sum()),
    1: int((y_train == 0).sum() * 1),
}
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_preprocessed, y_)

# --- Step 5: Train-test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)
# X_train_inb, X_test_inb, y_train_inb, y_test_inb = train_test_split(
#     X_resampled, y_resampled, test_size=0.2, random_state=42
# )
# --- Step 6: Train model on resampled data ---
model = LogisticRegression()
model.fit(X_train, y_train)

# --- Step 7: Final pipeline for inference (no SMOTE, just preprocessing + model) ---
inference_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

# --- Step 8: Predict on test set (untouched data) ---
# y_pred = inference_pipeline.predict(X_test)
# y_proba = inference_pipeline.predict_proba(X_test)[:, 1]

predictions_dict = {
    # "Test (Imbalanced)": {
    #     "y": y_train,
    #     "y_pred": inference_pipeline.predict(X_train),
    #     "y_prob": inference_pipeline.predict_proba(X_train)[:, 1]
    # },
    "Train (Balanced)": {
        "y": y_train,
        "y_pred": model.predict(X_train),
        "y_prob": model.predict_proba(X_train)[:, 1]
    },
    "Test (Balanced)": {
        "y": y_test,
        "y_pred": model.predict(X_test),
        "y_prob": model.predict_proba(X_test)[:, 1]
    },
    "Test (Inbalanced)": {
        "y": y_test_imbal,
        "y_pred": inference_pipeline.predict(X_test_imbal),
        "y_prob": inference_pipeline.predict_proba(X_test_imbal)[:, 1]
    },

}



print("Original training target distribution:", Counter(y_train))
print("Resampled training target distribution:", Counter(y_train_resampled))

metrics_df = calculate_model_metrics_datasets(predictions_dict)
metrics_df

In [ ]:
metrics_df_ = metrics_df[['Accuracy', 'F1 Score', 'Precision', 'Recall']].iloc[1:]
display(metrics_df_)

# sns.barplot(data=metrics_df_)

metrics_df_ = metrics_df_.reset_index(names='Data').melt(id_vars='Data')
# display(metrics_df_)

with sns.axes_style("whitegrid"):
    fig, axes = plt.subplots(figsize=(8,5))
    sns.barplot(metrics_df_, x='variable', y='value', hue='Data', ax=axes)
    axes.set_xlabel('')
    axes.set_ylabel('Metric value', fontsize=18)
    axes.legend(fontsize=14, ncol=2)
    axes.tick_params(axis='both', which='major', labelsize=14)
    axes.set_title("Metric for balanced and imbalanced test sets", fontsize=16)

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

print('')
print(f"accuracy_score: {accuracy_score(y_test, y_pred):.2f}")
print(f"recall_score: {recall_score(y_test, y_pred):.2f}")
print(f"f1_score: {f1_score(y_test, y_pred):.2f}")

print("\nTest data")
print(f"accuracy_score: {accuracy_score(y_test, y_pred):.2f}")
print(f"recall_score: {recall_score(y_test, y_pred):.2f}")
print(f"f1_score: {f1_score(y_test, y_pred):.2f}")


In [ ]:
# for rates in [.2, .4, .6, .8, 1]:

# X_train, X_test, y_train, y_test, X_train_preprocessed, X_train_resampled, y_train_resampled


    # solver='liblinear', 
    # class_weight='balanced', 
    # random_state=42


y_train_resampled

In [ ]:
# y_train.value_counts()
# y_train.value_counts().max()
print(f"{754 / (754+150):.2f}")
print(f"{754 / (754+150):.2f}")
print()

In [ ]:
# Start with the softbalss: EDA
print(f"X_train: {len(X_train)}")
print(f"X_train_preprocessed: {len(X_train_preprocessed)}")
print(f"X_train_resampled: {len(X_train_resampled)}")

print(f"y_train: {len(y_train)}")
print(f"y_train_resampled: {len(y_train_resampled)}")
print()
print(y_train.value_counts())
print()
print(y_train_resampled.value_counts())

# plot metrics on: preprocessed, resampled

In [ ]:
# 5% 20 40 60 80
# 5 20 35 50 65 80 95

# train on inbalanced data
# train on balanced data
# compare on train and TEST data

In [ ]:
# y_proc_df = pd.DataFrame(y_train_proc, columns=[target])
# min_ = y_proc_df.value_counts().min() # label 1
# sample_0 = y_proc_df[y_proc_df[target] == 0].sample(min_, random_state=42)
# sample_1 = y_proc_df[y_proc_df[target] == 1]

# index = list(sample_0.index) + list(sample_1.index)
# y_proc_df.loc[index]

# X_proc_resampled = X_train_proc[index]
# y_proc_resampled = y_train_proc[index]

# print(X_proc_resampled.shape)
# print(y_proc_resampled.shape)